### **Student Information**
GitHub ID: Hajermas

Kaggle name: hajermastouri

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**


## 1. Model Development (10 pts Required)




### 1.1 Preprocessing Steps

To prepare the social media dataset for emotion classification while preserving syntactic cues, a targeted preprocessing pipeline was implemented : 
- Data ingestion and JSON flattening : The raw corpus was flattened to extract post_id and text, and available hashtags. Hashtags were concatenated onto the main post body because they frequently carry explicit emotion-bearing terms that are direct cues for the six target emotions
- Dataset merging and split alignment : The flattened posts were partitioned into training and test sets. Ground-truth labels from emotion.csv were merged onto the training split while test_df was left-merged onto submission.csv to guarantee exact row ordering for kaggle submission.
- Affect-preserving social text normalization : Unlike topic classification like sports or religion in the first lab, aggressive noise removal damages emotion mining by stripping away psychological and syntactic signals. Therfore, clean_social_text normalizes only non-informative artifacts while preserving emotional markers :
     - HTML Decoding and Entity Masking : Decoded HTML entities (html.unescape) and replaced URLs and @mentions with placeholders so the models         do not overfit to specific links or usernames.
     - CamelCase hashtag splitting : Decomposed multi-word hashtags into separate words (#SoExcited to So Excited, or #SoNervous to So Nervous)         so word-level tokenizers and pretrained models can capture their full semantic meaning.
     - Punctuation and character elongation normalization : Separated repeated punctuation (!!, ??) into standalone tokens and capped character        elongations at two characters ("veryyy" / "veryyyy" to "veryy"). This prevents TF-IDF from treating expressive variations of the same           word as unrelated vocabulary items while retaining the emphasis cue.
     - Deliberate retention of stop Words: Standard stop-word removal was intentionally omitted. In emotion recognition, function words like           negations ("not", "never"), personal pronouns ("I", "my"), and interrogative particles ("how", "why", "what") provide essential                 syntactic and psychological clues (distinguishing surprise or fear from neutral statements). Uninformative high-frequency filler words          (such as "the" or "a") are instead naturally down-weighted during vectorization via inverse document frequency
- Deduplication : Exact duplicate (clean_text, label) pairs were removed from train_df (34 duplicates dropped, leaving 3,326 unique training posts) to prevent cross-validation leakage.



### 1.2 Feature Engineering Steps

To represent both explicit lexical markers and deeper contextual semantics across short social media posts, feature engineering was developed iteratively across two stages before converging on the final dual-branch representation : 

**Attempt 1 : Multi-Granularity TF-IDF + Handcrafted Stylistic Features :**
 -Word N-grams (1–3 grams): TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=25000, sublinear_tf=True) captured unigrams, bigrams,      and emotion-inverting phrases ("not happy", "so glad"). Sublinear term-frequency scaling (sublinear_tf=True) dampened repetitive high-          frequency filler words without discarding stop words.   
 -Character N-grams (3–5 chars within word boundaries): TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=35000, sublinear_tf=True) captured misspellings, slang, and expressive punctuation patterns ("!!", "??").   
 -Handcrafted stylistic and punctuation features: Nine domain-specific numerical features were extracted from raw_text and clean_text and         standardized inside each cross-validation fold via StandardScaler to prevent data leakage: character length (char_len), word count              (word_count), average word length (avg_word_len), exclamation count (excl_count), question mark count (quest_count), uppercase character        ratio (upper_ratio), all-caps word count (all_caps_count, capturing shouting emphasis), negation count (negation_count), and first-person       pronoun count (first_person_count)
 
**Attempt 2 : Dense Frozen Embeddings (all-MiniLM-L6-v2 :** 
L2-normalized 384-dimensional sentence embeddings from all-MiniLM-L6-v2 were horizontally stacked (scipy.sparse.hstack) onto the TF-IDF and stylistic features (60,393 total features). While this raised OOF Macro-F1 slightly from 0.5191 to 0.5531 (0.5482 to 0.5491 on the untuned blend), frozen general-purpose embeddings remained limited because their weights were never adapted to emotion boundaries and were overshadowed by high-dimensional sparse TF-IDF columns.

**Final Kaggle Submission Pipeline : Dual representation (Lexical/Stylistic + Subword BPE Tokenization) :**
Frozen MiniLM embeddings were dropped in favor of feeding two complementary feature representations into separate model branches:   
->Branch 1 (Sparse Lexical & Stylistic Matrix) : Word TF-IDF (1–3 grams) + Character TF-IDF (3–5 grams) + 9 fold-scaled stylistic features (StandardScaler), capturing exact keyword, subword, and punctuation cues.   
->Branch 2 (Contextual Transformer Tokenization): Raw post text (raw_text, truncated/padded to max_len=128) was tokenized using the byte-level BPE tokenizer of cardiffnlp/twitter-roberta-base-sentiment-latest, preserving casing, slang, and full-sentence token interactions for end-to-end fine-tuning on Kaggle GPU. 



### 1.3 Explanation of Your Model

To provide full transparency into the experimental progression and demonstrate how each architectural change impacted cross-validation performance, all three modeling iterations are intentionally preserved in the code sections below. 

Note on Kaggle GPU Execution: Because fine-tuning Twitter-RoBERTa across 5 folds required a dedicated GPU, the final submission pipeline was executed on Kaggle's GPU environment. While the full code is integrated into the corresponding sections of this notebook, the complete executed notebook with all cell outputs preserved can also be found in the repository under the file kaggle_code.ipynb. 

To address class imbalance across the six emotion categories and maximize the competition's Macro-F1 metric, the modeling strategy evolved iteratively across three stages using 5-fold stratified cross-validation :

**Stage 1** Classical TF-IDF & Stylistic Ensemble (OOF Macro-F1: 0.5191) :
The baseline model family combined Balanced Logistic Regression, Calibrated LinearSVC, and Complement Naive Bayes trained on word/character TF-IDF n-grams and standardized stylistic features. Using class-balanced weights and probability soft-voting (45% Logistic Regression, 40% LinearSVC, 15% Complement Naive Bayes), this ensemble captured explicit keywords, punctuation cues, and subword patterns while preventing majority classes like joy (F1: 0.6409) from dominating predictions.   

**Stage 2** Adding Frozen all-MiniLM-L6-v2 Embeddings (OOF Macro-F1: 0.5531):
In the second iteration, dense 384-dimensional sentence embeddings from a frozen all-MiniLM-L6-v2 encoder were concatenated with the TF-IDF and stylistic features before training the same linear ensemble. While adding semantic embeddings lifted OOF Macro-F1 from 0.5191 to 0.5531 (boosting joy to 0.6948 and fear to 0.5746), performance's growth could be explained because of MiniLM's weights that remained frozen and were never adapted to fine-grained emotion distinctions. 
Thus frozen MiniLM was dropped in favor of full end-to-end Transformer fine-tuning on Kaggle's GPU. 

**Stage 3** Fine-Tuned Twitter-RoBERTa and Final Hybrid Ensemble (OOF Macro-F1: 0.6040) :
To capture full sentence context, slang, and subtle emotional nuances, cardiffnlp/twitter-roberta-base-sentiment-latest was fine-tuned end to end on GPU (OOF Macro-F1: 0.5940 alone) using smoothed inverse square root class weights to lift recall on minority emotions. Finally, out-of-fold probabilities from the fine-tuned Twitter-RoBERTa (65%) and the Classical TF-IDF/Stylistic Ensemble (35%) were blended and post-processed with per-class Macro-F1 probability multipliers, reaching 0.6040 OOF Macro-F1 (0.6268 accuracy), 0.6201 on the public leaderboard, and 0.6319 on the private leaderboard.   
Error and performance analysis:
Across every modeling stage, joy consistently achieved the highest F1-score (rising from 0.6409 in Stage 1 to 0.7719 in Stage 3) due to having the largest training support (875 posts) and distinct positive vocabulary that stands out from the other emotions. By contrast, disgust consistently remained the lowest-scoring emotion across all iterations (0.3585 in Stage 1 to 0.4253 in Stage 2 to 0.4281 in Stage 3, with 0.3899 recall). The explanation for this could be its subtle lexical and contextual cues overlapping heavily with anger (0.5920 F1) and sadness (0.6471 F1) in short social media posts. Meanwhile, even though fear had the smallest sample size (288 posts), smoothed class weighting helped it reach 0.6182 F1 in the final ensemble.  



### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI


Throughout development, AI assistance was used to refine text normalization, feature extraction, error analysis, and model architecture:   

*-Stop-Word retention and character elongation normalization (Preprocessing):*

Prompt: "In preprocessing, do you recommend removing stopwords, and is standardizing elongated words like sooo/soooo to so helpful for 6-class emotion classification?"

AI Suggestion: Keep stop words intact (to preserve negations like "not", pronouns like "I"/"you", and question words) and collapse 3+ repeated letters to 2 characters ("veryyyy" to "veryy") rather than 1.   

Decision and why: Adopted. Removing stop words destroys negation polarity and syntactic emotion cues, while collapsing elongations to 2 characters groups sparse variants into a single vocabulary token without erasing the user's emphasis.   

*-Preserving uppercase intensity without vocabulary fragmentation (Feature Engineering):*

Prompt: "How can we spot uppercase letters when they preserve intensity while not treating the same words written with capital letters as different words? What is the trade-off?"   

AI Suggestion: Separate word identity from shouting intensity into two parallel channels rather than setting lowercase=False (which fragments a 3,326-row vocabulary): let TfidfVectorizer lowercase tokens (lowercase=True) so "Angry" and "angry" share statistical counts, while extracting explicit casing features (upper_ratio and all_caps_count) from raw_text (Option A) or inserting tags (Option B).   

Decision and why: Partially Adopted (Adopted Option A; Rejected Option B). Adding all_caps_count alongside upper_ratio in extract_stylistic_features captured shouting intensity cleanly without vocabulary fragmentation, whereas inline <CAPS> tagging (Option B) was skipped to keep clean_text clean for downstream tokenizers.   

*-Diagnosing low disgust F1-Score and label overlap (Error Analysis):*

Prompt: "Why did disgust get the lowest F1-score, and how can posts like post_id: 'expfslt' ('Takes a lot less effort to just say “fuck off, buy your own.”') be mistaken for anger?"

AI Suggestion: disgust suffers from both lower training support (454 posts) and strong lexical/pragmatic overlap with anger and sadness. Profanity, hostility, and dismissive expressions (such as "fuck off" in expfslt) appear across both anger and disgust annotations depending on subtle context (moral contempt vs. direct rage), causing linear and transformer classifiers to confuse the two. Use inverse-square-root class weighting and per-class threshold tuning to improve recall without flooding precision.   

Decision and why: Adopted. Applying smoothed class weights and Nelder-Mead threshold multipliers lifted disgust F1 from 0.3585 to 0.4281 while keeping precision balanced (0.4745).  

*-Replacing frozen MiniLM embeddings with end-to-end fine-tuned twitter-RoBERTa on GPU (Model Selection):*

Prompt: "Why did all-MiniLM-L6-v2 reach 0.55, why use RoBERTa specifically on GPU instead of fine-tuning MiniLM, and should we keep MiniLM when blending?"

AI Suggestion: all-MiniLM-L6-v2 was used as a frozen feature extractor trained only on general semantic similarity, so its weights never adapted to fine-grained emotions. Fine-tuning cardiffnlp/twitter-roberta-base-sentiment-latest on Kaggle's GPU leverages 124M parameters pretrained specifically on social media posts, and dropping frozen MiniLM simplifies the final blend with TF-IDF/stylistic features.

Decision and why: Adopted. Removing MiniLM and blending fine-tuned Twitter-RoBERTa (65%) with the TF-IDF/Stylistic ensemble (35%) simplified the pipeline and boosted the Kaggle score from 0.5531 to 0.6201 (Public) and 0.6319 (Private). 

*-Adding a second fine-tuned transformer (DeBERTa-v3-base) + stacking meta-learner (Post-submission exploration):*

Prompt: "Can we push 0.6201 even higher?"

AI Suggestion: Fine-tune a second Transformer architecture (microsoft/deberta-v3-base plus twitter-roberta-base-emotion-multilabel-latest) and train an out-of-fold Logistic Regression stacking meta-learner.   

Decision and why: Rejected for Final Submission. Due to Kaggle GPU session runtime constraints and competition deadline limits, the single 5-fold Twitter-RoBERTa + TF-IDF/Stylistic hybrid ensemble was retained as the final submission, achieving 0.6319 on the private leaderboard.   

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [7]:
### Add the code related to the preprocessing steps in cells inside this section
import json
import re
import html
import numpy as np
import pandas as pd
from pathlib import Path
# First attempts 
# Point directly to your kaggle_data folder in the repository root
DATA_DIR = Path("kaggle_data")

# Load the CSV metadata and submission template
df_ident = pd.read_csv(DATA_DIR / "data_identification.csv")
df_emotion = pd.read_csv(DATA_DIR / "emotion.csv")
df_sub = pd.read_csv(DATA_DIR / "sampleSubmission.csv")

print("data_identification:", df_ident.shape, df_ident.columns.tolist())
print("emotion:", df_emotion.shape, df_emotion.columns.tolist())
print("sampleSubmission:", df_sub.shape, df_sub.columns.tolist())

# Load and flatten final_posts.json into a DataFrame
with open(DATA_DIR / "final_posts.json", "r", encoding="utf-8") as f:
    raw_json = json.load(f)

df_posts = pd.json_normalize(raw_json)
print("Flattened JSON columns:", df_posts.columns.tolist())
print("Sample row:\n", df_posts.iloc[0].to_dict())

# Automatically detect the post_id, text, and optional hashtags columns
post_id_col = [c for c in df_posts.columns if c.endswith("post_id") or c.endswith("id")][0]
text_col = [c for c in df_posts.columns if c.endswith("text")][0]
hashtag_cols = [c for c in df_posts.columns if c.endswith("hashtags")]

df_posts_clean = pd.DataFrame({
    "post_id": df_posts[post_id_col],
    "text": df_posts[text_col].fillna(""),
    "hashtags": df_posts[hashtag_cols[0]].apply(
        lambda x: " ".join(x) if isinstance(x, list) else (str(x) if pd.notna(x) else "")
    ) if hashtag_cols else ""
})

# Merge posts with split identification and emotion labels
ident_id_col = df_ident.columns[0]
split_col = df_ident.columns[1]
emo_id_col = df_emotion.columns[0]
label_col = df_emotion.columns[1]

df_all = df_posts_clean.merge(df_ident, left_on="post_id", right_on=ident_id_col, how="right")
df_all = df_all.merge(df_emotion, left_on="post_id", right_on=emo_id_col, how="left")

df_all["raw_text"] = np.where(
    df_all["hashtags"].str.len() > 0,
    df_all["text"] + " " + df_all["hashtags"],
    df_all["text"]
)

# Social-media text normalization (preserves emotion cues while reducing noise)
def clean_social_text(text: str) -> str:
    text = html.unescape(str(text))
    text = re.sub(r"http\S+|www\.\S+", " <URL> ", text)         # normalize URLs
    text = re.sub(r"@\w+", " <USER> ", text)                     # normalize @mentions
    # Split CamelCase hashtags into separate words (e.g., #SoExcited -> So Excited)
    text = re.sub(r"#([A-Za-z0-9_]+)", lambda m: " " + re.sub(r"([a-z])([A-Z])", r"\1 \2", m.group(1)) + " ", text)
    text = re.sub(r"([!?.])\1+", r" \1\1 ", text)                # normalize repeated punctuation
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)                   # normalize character elongation
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_all["clean_text"] = df_all["raw_text"].apply(clean_social_text)

# Split into train_df and test_df 
train_df = df_all[df_all[split_col] == "train"].copy().reset_index(drop=True)
test_df = df_sub[["post_id"]].merge(
    df_all[df_all[split_col] == "test"], left_on="post_id", right_on="post_id", how="left"
)

# Drop exact duplicate (clean_text, label) pairs in train_df
before_dedup = len(train_df)
train_df = train_df.drop_duplicates(subset=["clean_text", label_col]).reset_index(drop=True)

print(f"\nTrain rows: {len(train_df)} (removed {before_dedup - len(train_df)} duplicates)")
print(f"Test rows:  {len(test_df)} (matches sampleSubmission: {len(test_df) == len(df_sub)})")
print("\nClass distribution:")
print(train_df[label_col].value_counts())

data_identification: (4800, 2) ['post_id', 'set']
emotion: (3360, 2) ['post_id', 'label']
sampleSubmission: (1440, 2) ['post_id', 'label']
Flattened JSON columns: ['post_id', 'text']
Sample row:
 {'post_id': 'f8bv5zh', 'text': 'Thought it looked weird as well.'}

Train rows: 3326 (removed 34 duplicates)
Test rows:  1440 (matches sampleSubmission: True)

Class distribution:
label
joy         870
anger       667
sadness     619
disgust     451
surprise    437
fear        282
Name: count, dtype: int64


In [ ]:
# Final attempt : submitted to kaggle
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

# Point to local repository dataset folder (use "/kaggle/input/competitions/2026-dm-lab-2-private-competition" when running on Kaggle)
DATA_DIR = Path("kaggle_data")

# Load tabular metadata and labels
sample_sub = pd.read_csv(DATA_DIR / "sampleSubmission.csv")
data_id = pd.read_csv(DATA_DIR / "data_identification.csv")
emotions = pd.read_csv(DATA_DIR / "emotion.csv")

print("sampleSubmission columns:", sample_sub.columns.tolist(), "shape:", sample_sub.shape)
print("data_identification columns:", data_id.columns.tolist(), "shape:", data_id.shape)
print("emotion columns:", emotions.columns.tolist(), "shape:", emotions.shape)

# Parse final_posts.json (handles list of dicts, nested {'root': {'_source': {'post': ...}}}, or {id: text} dict)
with open(DATA_DIR / "final_posts.json", "r", encoding="utf-8") as f:
    raw_json = json.load(f)

records = []
if isinstance(raw_json, dict):
    for pid, val in raw_json.items():
        if isinstance(val, str):
            records.append({"post_id": str(pid), "raw_text": val})
        elif isinstance(val, dict):
            text = val.get("text") or val.get("post_text") or val.get("content") or ""
            records.append({"post_id": str(val.get("post_id", val.get("id", pid))), "raw_text": text})
elif isinstance(raw_json, list):
    for item in raw_json:
        if isinstance(item, dict):
            node = item.get("root", item)
            node = node.get("_source", node) if isinstance(node, dict) else item
            node = node.get("post", node) if isinstance(node, dict) else item
            pid = node.get("post_id") or node.get("id") or item.get("post_id") or item.get("id")
            text = node.get("text") or node.get("post_text") or node.get("content") or ""
            records.append({"post_id": str(pid), "raw_text": str(text)})

posts_df = pd.DataFrame(records).drop_duplicates(subset=["post_id"])
print("Parsed posts_df shape:", posts_df.shape)

# Standardize ID column names & merge train / test splits
id_col_data = data_id.columns[0]
split_col = data_id.columns[1]
data_id[id_col_data] = data_id[id_col_data].astype(str)

id_col_emo = emotions.columns[0]
label_col = emotions.columns[1]
emotions[id_col_emo] = emotions[id_col_emo].astype(str)

merged = data_id.merge(posts_df, left_on=id_col_data, right_on="post_id", how="left")
merged["raw_text"] = merged["raw_text"].fillna("")

train_df = (
    merged[merged[split_col].str.lower() == "train"]
    .merge(emotions, left_on=id_col_data, right_on=id_col_emo, how="inner")
    .reset_index(drop=True)
)

# Preserve exact sampleSubmission row order for test_df
sub_id_col, sub_target_col = sample_sub.columns[0], sample_sub.columns[1]
sample_sub[sub_id_col] = sample_sub[sub_id_col].astype(str)
test_df = sample_sub[[sub_id_col]].merge(
    merged, left_on=sub_id_col, right_on=id_col_data, how="left"
).reset_index(drop=True)
test_df["raw_text"] = test_df["raw_text"].fillna("")

# Social media text cleaning (preserves emotional cues, splits CamelCase hashtags, separates punctuation)
def clean_social_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    # Split hashtags like #SoGross -> So Gross
    text = re.sub(r"#([A-Za-z0-9_]+)", lambda m: " " + re.sub(r"([a-z])([A-Z])", r"\1 \2", m.group(1)) + " ", text)
    # Normalize repeated characters beyond 2 (e.g. "sooooo" -> "soo")
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    # Separate exclamation/question marks so they become standalone tokens
    text = re.sub(r"([!?]+)", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df["raw_text"].apply(clean_social_text)
test_df["clean_text"] = test_df["raw_text"].apply(clean_social_text)

print(f"\nTrain shape: {train_df.shape} | Test shape: {test_df.shape}")
print("Class distribution:\n", train_df[label_col].value_counts())

## 2. Feature Engineering Steps

In [4]:
### Add the code related to the feature engineering steps in cells inside this section


#1st attempt : TF-IDF + Stylistic baseline

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler



# Handcrafted Stylistic and Punctuation Features
def extract_stylistic_features(df: pd.DataFrame) -> pd.DataFrame:
    raw = df["raw_text"].fillna("")
    clean = df["clean_text"].fillna("")
    feats = pd.DataFrame(index=df.index)
    feats["char_len"] = raw.str.len()
    feats["word_count"] = clean.str.split().str.len()
    feats["avg_word_len"] = feats["char_len"] / (feats["word_count"] + 1)
    feats["excl_count"] = raw.str.count(r"!")
    feats["quest_count"] = raw.str.count(r"\?")
    feats["upper_ratio"] = raw.apply(lambda s: sum(1 for c in s if c.isupper()) / (len(s) + 1))
    feats["negation_count"] = clean.str.lower().str.count(r"\b(not|no|never|nothing|nowhere|neither|nobody|none|n't)\b")
    feats["first_person_count"] = clean.str.lower().str.count(r"\b(i|me|my|mine|myself|we|us|our)\b")
    return feats

X_style_train_df = extract_stylistic_features(train_df)
X_style_test_df = extract_stylistic_features(test_df)

scaler = StandardScaler()
X_style_train = csr_matrix(scaler.fit_transform(X_style_train_df))
X_style_test = csr_matrix(scaler.transform(X_style_test_df))

# Word N-gram TF-IDF (1 to 3 grams, sublinear TF scaling)
word_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    min_df=2,
    max_features=25000,
    sublinear_tf=True,
    strip_accents="unicode"
)
X_word_train = word_vec.fit_transform(train_df["clean_text"])
X_word_test = word_vec.transform(test_df["clean_text"])

# Character N-gram TF-IDF (3 to 5 chars within word boundaries)
char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_features=35000,
    sublinear_tf=True
)
X_char_train = char_vec.fit_transform(train_df["clean_text"])
X_char_test = char_vec.transform(test_df["clean_text"])

# Combine TF-IDF (Word + Char) and Stylistic features
X_train_tfidf = hstack([X_word_train, X_char_train, X_style_train]).tocsr()
X_test_tfidf = hstack([X_word_test, X_char_test, X_style_test]).tocsr()

print(f"Word TF-IDF shape:      {X_word_train.shape}")
print(f"Char TF-IDF shape:      {X_char_train.shape}")
print(f"Stylistic feats shape:  {X_style_train.shape}")
print(f"Combined TF-IDF matrix: {X_train_tfidf.shape}")

Word TF-IDF shape:      (3326, 8625)
Char TF-IDF shape:      (3326, 18855)
Stylistic feats shape:  (3326, 8)
Combined TF-IDF matrix: (3326, 27488)


In [8]:
#2nd attempt: TF-IDF + MiniLM

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

# Handcrafted Stylistic and Punctuation Feature Extractor (9 features)
def extract_stylistic_features(df: pd.DataFrame) -> pd.DataFrame:
    raw = df["raw_text"].fillna("")
    clean = df["clean_text"].fillna("")
    feats = pd.DataFrame(index=df.index)
    
    feats["char_len"] = raw.str.len()
    feats["word_count"] = clean.str.split().str.len()
    feats["avg_word_len"] = feats["char_len"] / (feats["word_count"] + 1)
    feats["excl_count"] = raw.str.count(r"!")
    feats["quest_count"] = raw.str.count(r"\?")
    feats["upper_ratio"] = raw.apply(lambda s: sum(1 for c in s if c.isupper()) / (len(s) + 1))
    feats["all_caps_count"] = raw.str.count(r"\b[A-Z]{2,}\b")  # captures shouting (e.g., "WHAT", "NO")
    feats["negation_count"] = clean.str.lower().str.count(
        r"\b(not|no|never|nothing|nowhere|neither|nobody|none|n't)\b"
    )
    feats["first_person_count"] = clean.str.lower().str.count(
        r"\b(i|me|my|mine|myself|we|us|our)\b"
    )
    return feats

X_style_train_raw = extract_stylistic_features(train_df)
X_style_test_raw = extract_stylistic_features(test_df)

# Precompute L2-Normalized Dense Contextual Embeddings (all-MiniLM-L6-v2, 384-dim)
# (Frozen pretrained encoder — safe to compute once before CV)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
X_emb_train = embedder.encode(
    train_df["raw_text"].tolist(), show_progress_bar=False, normalize_embeddings=True
)
X_emb_test = embedder.encode(
    test_df["raw_text"].tolist(), show_progress_bar=False, normalize_embeddings=True
)

# Helper to build leak-free TF-IDF + Scaled Stylistic + MiniLM matrices inside each CV fold
def build_fold_features(tr_idx, va_idx):
    # Scale stylistic features strictly on training fold
    scaler = StandardScaler()
    st_tr = csr_matrix(scaler.fit_transform(X_style_train_raw.iloc[tr_idx]))
    st_va = csr_matrix(scaler.transform(X_style_train_raw.iloc[va_idx]))
    st_te = csr_matrix(scaler.transform(X_style_test_raw))

    # Word N-grams (1-3) fitted strictly on training fold
    word_vec = TfidfVectorizer(
        ngram_range=(1, 3), min_df=2, max_features=25000,
        sublinear_tf=True, strip_accents="unicode"
    )
    w_tr = word_vec.fit_transform(train_df["clean_text"].iloc[tr_idx])
    w_va = word_vec.transform(train_df["clean_text"].iloc[va_idx])
    w_te = word_vec.transform(test_df["clean_text"])

    # Character N-grams (3-5) fitted strictly on training fold
    char_vec = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), min_df=3,
        max_features=35000, sublinear_tf=True
    )
    c_tr = char_vec.fit_transform(train_df["clean_text"].iloc[tr_idx])
    c_va = char_vec.transform(train_df["clean_text"].iloc[va_idx])
    c_te = char_vec.transform(test_df["clean_text"])

    # Non-negative TF-IDF-only matrices for ComplementNB
    X_tr_nb = hstack([w_tr, c_tr]).tocsr()
    X_va_nb = hstack([w_va, c_va]).tocsr()
    X_te_nb = hstack([w_te, c_te]).tocsr()

    # Full Fused Matrices (Word TF-IDF + Char TF-IDF + Stylistic + MiniLM Embeddings)
    X_tr_fused = hstack([w_tr, c_tr, st_tr, csr_matrix(X_emb_train[tr_idx])]).tocsr()
    X_va_fused = hstack([w_va, c_va, st_va, csr_matrix(X_emb_train[va_idx])]).tocsr()
    X_te_fused = hstack([w_te, c_te, st_te, csr_matrix(X_emb_test)]).tocsr()

    return (X_tr_fused, X_va_fused, X_te_fused), (X_tr_nb, X_va_nb, X_te_nb)

print(f"Stylistic features extracted: {X_style_train_raw.shape[1]} columns")
print(f"MiniLM embeddings computed:   {X_emb_train.shape}")
print("Feature engineering pipeline ready for 5-Fold Stratified CV.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Stylistic features extracted: 9 columns
MiniLM embeddings computed:   (3326, 384)
Feature engineering pipeline ready for 5-Fold Stratified CV.


In [ ]:
# Final attempt : submitted to kaggle 
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

# Handcrafted Stylistic & Punctuation Features (9 features)
def extract_stylistic_features(df: pd.DataFrame) -> pd.DataFrame:
    raw = df["raw_text"].fillna("")
    clean = df["clean_text"].fillna("")
    feats = pd.DataFrame(index=df.index)
    feats["char_len"] = raw.str.len()
    feats["word_count"] = clean.str.split().str.len()
    feats["avg_word_len"] = feats["char_len"] / (feats["word_count"] + 1)
    feats["excl_count"] = raw.str.count(r"!")
    feats["quest_count"] = raw.str.count(r"\?")
    feats["upper_ratio"] = raw.apply(lambda s: sum(1 for c in s if c.isupper()) / (len(s) + 1))
    feats["all_caps_count"] = raw.str.count(r"\b[A-Z]{2,}\b")
    feats["negation_count"] = clean.str.lower().str.count(
        r"\b(not|no|never|nothing|nowhere|neither|nobody|none|n't)\b"
    )
    feats["first_person_count"] = clean.str.lower().str.count(
        r"\b(i|me|my|mine|myself|we|us|our)\b"
    )
    return feats

X_style_train_raw = extract_stylistic_features(train_df)
X_style_test_raw = extract_stylistic_features(test_df)

# Leak-free fold-level builder for Word TF-IDF (1-3), Char TF-IDF (3-5), and Scaled Stylistic features
def build_lexical_fold_features(tr_idx, va_idx):
    scaler = StandardScaler()
    st_tr = csr_matrix(scaler.fit_transform(X_style_train_raw.iloc[tr_idx]))
    st_va = csr_matrix(scaler.transform(X_style_train_raw.iloc[va_idx]))
    st_te = csr_matrix(scaler.transform(X_style_test_raw))

    word_vec = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=25000, sublinear_tf=True, strip_accents="unicode")
    w_tr = word_vec.fit_transform(train_df["clean_text"].iloc[tr_idx])
    w_va = word_vec.transform(train_df["clean_text"].iloc[va_idx])
    w_te = word_vec.transform(test_df["clean_text"])

    char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=35000, sublinear_tf=True)
    c_tr = char_vec.fit_transform(train_df["clean_text"].iloc[tr_idx])
    c_va = char_vec.transform(train_df["clean_text"].iloc[va_idx])
    c_te = char_vec.transform(test_df["clean_text"])

    X_tr_nb, X_va_nb, X_te_nb = hstack([w_tr, c_tr]).tocsr(), hstack([w_va, c_va]).tocsr(), hstack([w_te, c_te]).tocsr()
    X_tr_feat = hstack([w_tr, c_tr, st_tr]).tocsr()
    X_va_feat = hstack([w_va, c_va, st_va]).tocsr()
    X_te_feat = hstack([w_te, c_te, st_te]).tocsr()

    return (X_tr_feat, X_va_feat, X_te_feat), (X_tr_nb, X_va_nb, X_te_nb)

# Subword BPE Tokenization & PyTorch Dataset for Twitter-RoBERTa
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True, max_length=max_len, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

print(f"Stylistic features ready: {X_style_train_raw.shape[1]} columns")
print(f"Tokenizer initialized: {MODEL_NAME}")

## 3. Model Implementation Steps

In [6]:
### Add the code related to the model implementation steps in cells inside this section
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize

CLASSES = sorted(train_df[label_col].unique())
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for i, c in enumerate(CLASSES)}
y_train = train_df[label_col].map(label2id).values

 
X_train_tfidf_nonneg = hstack([X_word_train, X_char_train]).tocsr()
X_test_tfidf_nonneg = hstack([X_word_test, X_char_test]).tocsr()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lr = np.zeros((len(train_df), len(CLASSES)))
test_lr = np.zeros((len(test_df), len(CLASSES)))

oof_svc = np.zeros((len(train_df), len(CLASSES)))
test_svc = np.zeros((len(test_df), len(CLASSES)))

oof_nb = np.zeros((len(train_df), len(CLASSES)))
test_nb = np.zeros((len(test_df), len(CLASSES)))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_tfidf, y_train), 1):
    # Model A: Balanced Logistic Regression
    clf_lr = LogisticRegression(C=1.5, class_weight="balanced", max_iter=1000, random_state=42)
    clf_lr.fit(X_train_tfidf[tr_idx], y_train[tr_idx])
    oof_lr[va_idx] = clf_lr.predict_proba(X_train_tfidf[va_idx])
    test_lr += clf_lr.predict_proba(X_test_tfidf) / skf.n_splits

    # Model B: Calibrated LinearSVC
    base_svc = LinearSVC(C=0.35, class_weight="balanced", random_state=42)
    clf_svc = CalibratedClassifierCV(base_svc, cv=3)
    clf_svc.fit(X_train_tfidf[tr_idx], y_train[tr_idx])
    oof_svc[va_idx] = clf_svc.predict_proba(X_train_tfidf[va_idx])
    test_svc += clf_svc.predict_proba(X_test_tfidf) / skf.n_splits

    # Model C: Complement Naive Bayes (designed specifically for imbalanced text data)
    clf_nb = ComplementNB(alpha=0.5)
    clf_nb.fit(X_train_tfidf_nonneg[tr_idx], y_train[tr_idx])
    oof_nb[va_idx] = clf_nb.predict_proba(X_train_tfidf_nonneg[va_idx])
    test_nb += clf_nb.predict_proba(X_test_tfidf_nonneg) / skf.n_splits

print("5-Fold OOF Macro-F1 Scores (TF-IDF Baseline):")
print(f"  Model A (Balanced LogisticRegression): {f1_score(y_train, oof_lr.argmax(axis=1), average='macro'):.4f}")
print(f"  Model B (Calibrated LinearSVC):        {f1_score(y_train, oof_svc.argmax(axis=1), average='macro'):.4f}")
print(f"  Model C (Complement Naive Bayes):      {f1_score(y_train, oof_nb.argmax(axis=1), average='macro'):.4f}")

# Soft-voting ensemble blend
oof_ens = 0.45 * oof_lr + 0.35 * oof_svc + 0.20 * oof_nb
test_ens = 0.45 * test_lr + 0.35 * test_svc + 0.20 * test_nb
raw_ens_f1 = f1_score(y_train, oof_ens.argmax(axis=1), average="macro")
print(f"  Soft-Voting Ensemble (raw argmax):     {raw_ens_f1:.4f}")

# Optimize class multipliers on OOF probabilities to maximize Macro-F1
def loss_func(weights):
    scaled_probs = oof_ens * weights
    return -f1_score(y_train, scaled_probs.argmax(axis=1), average="macro")

res = minimize(loss_func, np.ones(len(CLASSES)), method="Nelder-Mead", options={"maxiter": 500})
best_weights = res.x if -res.fun > raw_ens_f1 else np.ones(len(CLASSES))
opt_oof_preds = (oof_ens * best_weights).argmax(axis=1)
print(f"  Soft-Voting Ensemble (F1-tuned):       {f1_score(y_train, opt_oof_preds, average='macro'):.4f}\n")

print("Out-of-Fold Classification Report:")
print(classification_report(y_train, opt_oof_preds, target_names=CLASSES, digits=4))

# Save Kaggle submission
final_test_preds = [id2label[i] for i in (test_ens * best_weights).argmax(axis=1)]
submission = pd.DataFrame({
    "id": test_df["post_id"],
    "label": final_test_preds
})
submission.to_csv("submission.csv", index=False)

print("Saved submission.csv! Shape:", submission.shape)
print("Predicted test label distribution:")
print(submission["label"].value_counts())

C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increas

5-Fold OOF Macro-F1 Scores (TF-IDF Baseline):
  Model A (Balanced LogisticRegression): 0.4989
  Model B (Calibrated LinearSVC):        0.5025
  Model C (Complement Naive Bayes):      0.5045
  Soft-Voting Ensemble (raw argmax):     0.5123
  Soft-Voting Ensemble (F1-tuned):       0.5191

Out-of-Fold Classification Report:
              precision    recall  f1-score   support

       anger     0.5036    0.5172    0.5104       667
     disgust     0.4050    0.3215    0.3585       451
        fear     0.6649    0.4504    0.5370       282
         joy     0.6161    0.6678    0.6409       870
     sadness     0.5101    0.6090    0.5552       619
    surprise     0.5293    0.4966    0.5124       437

    accuracy                         0.5388      3326
   macro avg     0.5382    0.5104    0.5191      3326
weighted avg     0.5379    0.5388    0.5348      3326

Saved submission.csv! Shape: (1440, 2)
Predicted test label distribution:
label
joy         427
sadness     316
anger       303
surpris

In [9]:
#2nd attempt: TF-IDF + MiniLM
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize

# Encode target labels
CLASSES = sorted(train_df[label_col].unique())
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for i, c in enumerate(CLASSES)}
y_train = train_df[label_col].map(label2id).values

# Initialize Out-of-Fold (OOF) and Test prediction arrays
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lr = np.zeros((len(train_df), len(CLASSES)))
test_lr = np.zeros((len(test_df), len(CLASSES)))

oof_svc = np.zeros((len(train_df), len(CLASSES)))
test_svc = np.zeros((len(test_df), len(CLASSES)))

oof_nb = np.zeros((len(train_df), len(CLASSES)))
test_nb = np.zeros((len(test_df), len(CLASSES)))

# 5-Fold Stratified Cross-Validation Loop
for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_train), 1):
    (X_tr_fused, X_va_fused, X_te_fused), (X_tr_nb, X_va_nb, X_te_nb) = build_fold_features(tr_idx, va_idx)

    # Model A: Balanced Logistic Regression on Fused Features
    clf_lr = LogisticRegression(C=1.5, class_weight="balanced", max_iter=1000, random_state=42)
    clf_lr.fit(X_tr_fused, y_train[tr_idx])
    oof_lr[va_idx] = clf_lr.predict_proba(X_va_fused)
    test_lr += clf_lr.predict_proba(X_te_fused) / skf.n_splits

    # Model B: Calibrated LinearSVC on Fused Features
    base_svc = LinearSVC(C=0.35, class_weight="balanced", random_state=42)
    clf_svc = CalibratedClassifierCV(base_svc, cv=3)
    clf_svc.fit(X_tr_fused, y_train[tr_idx])
    oof_svc[va_idx] = clf_svc.predict_proba(X_va_fused)
    test_svc += clf_svc.predict_proba(X_te_fused) / skf.n_splits

    # Model C: Complement Naive Bayes on Non-Negative TF-IDF Features
    clf_nb = ComplementNB(alpha=0.5)
    clf_nb.fit(X_tr_nb, y_train[tr_idx])
    oof_nb[va_idx] = clf_nb.predict_proba(X_va_nb)
    test_nb += clf_nb.predict_proba(X_te_nb) / skf.n_splits

    fold_blend = 0.45 * oof_lr[va_idx] + 0.40 * oof_svc[va_idx] + 0.15 * oof_nb[va_idx]
    print(f"Fold {fold} Macro-F1: {f1_score(y_train[va_idx], fold_blend.argmax(axis=1), average='macro'):.4f}")

# Evaluate Individual Models and Soft-Voting Ensemble
print("\n5-Fold OOF Macro-F1 Summary (Hybrid TF-IDF + Style + MiniLM):")
print(f"  Model A (Balanced LogisticRegression): {f1_score(y_train, oof_lr.argmax(axis=1), average='macro'):.4f}")
print(f"  Model B (Calibrated LinearSVC):        {f1_score(y_train, oof_svc.argmax(axis=1), average='macro'):.4f}")
print(f"  Model C (Complement Naive Bayes):      {f1_score(y_train, oof_nb.argmax(axis=1), average='macro'):.4f}")

oof_ens = 0.45 * oof_lr + 0.40 * oof_svc + 0.15 * oof_nb
test_ens = 0.45 * test_lr + 0.40 * test_svc + 0.15 * test_nb
raw_ens_f1 = f1_score(y_train, oof_ens.argmax(axis=1), average="macro")
print(f"  Soft-Voting Ensemble (raw argmax):     {raw_ens_f1:.4f}")

# Optimize Per-Class Multipliers for Macro-F1
def macro_f1_loss(weights):
    return -f1_score(y_train, (oof_ens * weights).argmax(axis=1), average="macro")

res = minimize(macro_f1_loss, np.ones(len(CLASSES)), method="Nelder-Mead", options={"maxiter": 600})
best_weights = res.x if -res.fun > raw_ens_f1 else np.ones(len(CLASSES))

opt_oof_preds = (oof_ens * best_weights).argmax(axis=1)
final_oof_f1 = f1_score(y_train, opt_oof_preds, average="macro")
print(f"  Soft-Voting Ensemble (F1-tuned):       {final_oof_f1:.4f}  (vs TF-IDF Baseline: 0.5191)\n")

print("Out-of-Fold Classification Report:")
print(classification_report(y_train, opt_oof_preds, target_names=CLASSES, digits=4))

# Generate Kaggle Submission File ('id' and 'label' columns)
final_test_preds = [id2label[i] for i in (test_ens * best_weights).argmax(axis=1)]
submission = pd.DataFrame({
    "id": test_df["post_id"].values,
    "label": final_test_preds
})
submission.to_csv("submission.csv", index=False)

print(f"Saved submission.csv! Shape: {submission.shape}")
print("Predicted test label distribution:")
print(submission["label"].value_counts())

C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fold 1 Macro-F1: 0.5938


C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fold 2 Macro-F1: 0.5488


C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fold 3 Macro-F1: 0.5439


C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fold 4 Macro-F1: 0.5425


C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\dell\Documents\DM2026Labs\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fold 5 Macro-F1: 0.5059

5-Fold OOF Macro-F1 Summary (Hybrid TF-IDF + Style + MiniLM):
  Model A (Balanced LogisticRegression): 0.5395
  Model B (Calibrated LinearSVC):        0.5437
  Model C (Complement Naive Bayes):      0.5122
  Soft-Voting Ensemble (raw argmax):     0.5471
  Soft-Voting Ensemble (F1-tuned):       0.5531  (vs TF-IDF Baseline: 0.5191)

Out-of-Fold Classification Report:
              precision    recall  f1-score   support

       anger     0.5188    0.5382    0.5283       667
     disgust     0.4956    0.3725    0.4253       451
        fear     0.6063    0.5461    0.5746       282
         joy     0.6484    0.7483    0.6948       870
     sadness     0.5659    0.5687    0.5673       619
    surprise     0.5422    0.5149    0.5282       437

    accuracy                         0.5740      3326
   macro avg     0.5629    0.5481    0.5531      3326
weighted avg     0.5688    0.5740    0.5691      3326

Saved submission.csv! Shape: (1440, 2)
Predicted test label dist

In [ ]:
# Final attempt implementation : submitted to kaggle 
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize

# Encode target labels & set up 5-Fold Stratified CV
CLASSES = sorted(train_df[label_col].unique())
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for i, c in enumerate(CLASSES)}
y_train = train_df[label_col].map(label2id).values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model Family 1: Classical Lexical & Stylistic Soft-Voting Ensemble
oof_lr = np.zeros((len(train_df), len(CLASSES)))
test_lr = np.zeros((len(test_df), len(CLASSES)))
oof_svc = np.zeros((len(train_df), len(CLASSES)))
test_svc = np.zeros((len(test_df), len(CLASSES)))
oof_nb = np.zeros((len(train_df), len(CLASSES)))
test_nb = np.zeros((len(test_df), len(CLASSES)))

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_train), 1):
    (X_tr_feat, X_va_feat, X_te_feat), (X_tr_nb, X_va_nb, X_te_nb) = build_lexical_fold_features(tr_idx, va_idx)

    clf_lr = LogisticRegression(C=1.5, class_weight="balanced", max_iter=1000, random_state=42)
    clf_lr.fit(X_tr_feat, y_train[tr_idx])
    oof_lr[va_idx] = clf_lr.predict_proba(X_va_feat)
    test_lr += clf_lr.predict_proba(X_te_feat) / skf.n_splits

    clf_svc = CalibratedClassifierCV(LinearSVC(C=0.35, class_weight="balanced", random_state=42), cv=3)
    clf_svc.fit(X_tr_feat, y_train[tr_idx])
    oof_svc[va_idx] = clf_svc.predict_proba(X_va_feat)
    test_svc += clf_svc.predict_proba(X_te_feat) / skf.n_splits

    clf_nb = ComplementNB(alpha=0.5)
    clf_nb.fit(X_tr_nb, y_train[tr_idx])
    oof_nb[va_idx] = clf_nb.predict_proba(X_va_nb)
    test_nb += clf_nb.predict_proba(X_te_nb) / skf.n_splits

oof_ens = 0.45 * oof_lr + 0.40 * oof_svc + 0.15 * oof_nb
test_ens = 0.45 * test_lr + 0.40 * test_svc + 0.15 * test_nb
print(f"Model Family 1 (TF-IDF + Stylistic) 5-Fold OOF Macro-F1: {f1_score(y_train, oof_ens.argmax(axis=1), average='macro'):.4f}")

# Model Family 2: 5-Fold GPU Fine-Tuning of Twitter-RoBERTa
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running Transformer fine-tuning on:", device)
assert device.type == "cuda", "Turn on GPU T4/P100 before running Transformer fine-tuning!"

test_ds = EmotionDataset(test_df["raw_text"].fillna(""))
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

oof_trf = np.zeros((len(train_df), len(CLASSES)))
test_trf = np.zeros((len(test_df), len(CLASSES)))

EPOCHS = 4
BATCH_SIZE = 32
LR = 2e-5

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_train), 1):
    torch.manual_seed(42 + fold)
    np.random.seed(42 + fold)

    train_ds = EmotionDataset(train_df["raw_text"].iloc[tr_idx].fillna(""), y_train[tr_idx])
    val_ds = EmotionDataset(train_df["raw_text"].iloc[va_idx].fillna(""), y_train[va_idx])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    # Smooth inverse-sqrt class weights to lift minority classes (disgust, fear)
    raw_cw = compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=y_train[tr_idx])
    smooth_cw = torch.tensor(np.sqrt(raw_cw), dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=smooth_cw, label_smoothing=0.05)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(CLASSES), ignore_mismatched_sizes=True
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )

    best_val_f1 = -1.0
    best_val_probs = None
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            outputs = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            loss = criterion(outputs.logits, batch["labels"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        model.eval()
        val_probs = []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                ).logits
                val_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())

        val_probs = np.vstack(val_probs)
        val_f1 = f1_score(y_train[va_idx], val_probs.argmax(axis=1), average="macro")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_probs = val_probs
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    oof_trf[va_idx] = best_val_probs
    print(f"Fold {fold} Best Twitter-RoBERTa Macro-F1: {best_val_f1:.4f}")

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    fold_test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            ).logits
            fold_test_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())

    test_trf += np.vstack(fold_test_probs) / skf.n_splits

    del model, optimizer, scheduler, best_state
    torch.cuda.empty_cache()
    gc.collect()

# Hybrid Ensemble Blend (Twitter-RoBERTa + Classical TF-IDF/Stylistic)
trf_oof_f1 = f1_score(y_train, oof_trf.argmax(axis=1), average="macro")
print(f"\nModel Family 2 (Twitter-RoBERTa alone) 5-Fold OOF Macro-F1: {trf_oof_f1:.4f}")

best_alpha, best_blend_f1 = 0.80, -1.0
for alpha in [0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    blend_probs = alpha * oof_trf + (1 - alpha) * oof_ens
    score = f1_score(y_train, blend_probs.argmax(axis=1), average="macro")
    if score > best_blend_f1:
        best_blend_f1 = score
        best_alpha = alpha

oof_final = best_alpha * oof_trf + (1 - best_alpha) * oof_ens
test_final = best_alpha * test_trf + (1 - best_alpha) * test_ens
print(f"Blended Ensemble ({best_alpha:.2f} RoBERTa + {1-best_alpha:.2f} Hybrid) OOF Macro-F1: {best_blend_f1:.4f}")

# Per-Class Threshold Multiplier Tuning for Macro-F1
res = minimize(
    lambda w: -f1_score(y_train, (oof_final * w).argmax(axis=1), average="macro"),
    np.ones(len(CLASSES)),
    method="Nelder-Mead",
    options={"maxiter": 600},
)
best_weights = res.x if -res.fun > best_blend_f1 else np.ones(len(CLASSES))

opt_oof_preds = (oof_final * best_weights).argmax(axis=1)
final_oof_f1 = f1_score(y_train, opt_oof_preds, average="macro")
print(f"Final F1-Tuned Ensemble OOF Macro-F1: {final_oof_f1:.4f}\n")
print(classification_report(y_train, opt_oof_preds, target_names=CLASSES, digits=4))

# Export Kaggle Submission with explicit 'id,label' header
final_test_preds = [id2label[i] for i in (test_final * best_weights).argmax(axis=1)]
submission = pd.DataFrame({
    "id": sample_sub[sub_id_col].values,
    "label": final_test_preds,
})
assert list(submission.columns) == ["id", "label"]
assert len(submission) == len(sample_sub)
assert submission["label"].isna().sum() == 0

submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv! First 5 rows:")
print(submission.head())